# Q5: What is the process behind the Non-Max Suppression method?
**Topic:** Computer-vision | **Difficulty:** Medium | **Time:** 8 min
**Sheet:** Grind75ML

---

## Question
What is the process behind the Non-Max Suppression method?

## Detailed Answer

**Non-Max Suppression (NMS)** is a post-processing algorithm used in object detection to eliminate redundant overlapping bounding boxes, keeping only the most confident detection per object.

### Why NMS is Needed
Object detectors (YOLO, SSD, Faster R-CNN) produce many overlapping bounding boxes for the same object. NMS filters these down to one box per object.

### Algorithm Step-by-Step:

```
Input: List of boxes B with confidence scores S, IoU threshold T

1. Sort B by confidence scores (descending)
2. Select the box with highest score → add to KEEP list
3. Compute IoU of this box with all remaining boxes
4. Remove all boxes with IoU > threshold T (they detect the same object)
5. Repeat steps 2-4 until no boxes remain

Output: KEEP list (one box per detected object)
```

### IoU (Intersection over Union):
$$IoU(A, B) = \frac{\text{Area}(A \cap B)}{\text{Area}(A \cup B)}$$

Typical threshold: **0.45–0.5** (boxes with IoU > threshold are considered duplicates)

In [ ]:
import numpy as np


def compute_iou(box1: np.ndarray, box2: np.ndarray) -> float:
    """Compute IoU between two boxes [x1, y1, x2, y2]."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0.0


def nms(boxes: np.ndarray, scores: np.ndarray, iou_threshold: float = 0.5) -> list[int]:
    """Non-Max Suppression.
    
    Args:
        boxes: Array of shape (N, 4) with [x1, y1, x2, y2] format.
        scores: Confidence scores of shape (N,).
        iou_threshold: Boxes with IoU > threshold are suppressed.
    
    Returns:
        List of indices of kept boxes.
    """
    # Sort by confidence (descending)
    order = np.argsort(scores)[::-1]
    keep = []
    
    while len(order) > 0:
        # Pick the highest confidence box
        idx = order[0]
        keep.append(idx)
        
        if len(order) == 1:
            break
        
        # Compute IoU with all remaining boxes
        remaining = order[1:]
        ious = np.array([compute_iou(boxes[idx], boxes[r]) for r in remaining])
        
        # Keep only boxes with IoU <= threshold
        mask = ious <= iou_threshold
        order = remaining[mask]
    
    return keep

In [ ]:
# --- Example ---
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Simulated detections for one object (overlapping boxes)
boxes = np.array([
    [100, 100, 250, 300],  # high conf
    [110, 105, 260, 310],  # overlapping
    [105, 98,  255, 305],  # overlapping
    [400, 200, 520, 380],  # different object
    [410, 210, 530, 390],  # overlapping with above
], dtype=float)
scores = np.array([0.95, 0.80, 0.70, 0.90, 0.60])

keep_indices = nms(boxes, scores, iou_threshold=0.5)

print(f"Before NMS: {len(boxes)} boxes")
print(f"After NMS:  {len(keep_indices)} boxes (indices: {keep_indices})")
print(f"Kept scores: {scores[keep_indices]}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.Set1(np.linspace(0, 1, len(boxes)))
for ax, title, indices in [
    (axes[0], 'Before NMS', range(len(boxes))),
    (axes[1], 'After NMS', keep_indices)
]:
    ax.set_xlim(0, 600)
    ax.set_ylim(0, 500)
    ax.invert_yaxis()
    ax.set_title(title, fontsize=14)
    for i in indices:
        x1, y1, x2, y2 = boxes[i]
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                 edgecolor=colors[i], facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-5, f'{scores[i]:.2f}', color=colors[i], fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## Variants of NMS

| Variant | How it differs | Use case |
|---------|---------------|----------|
| **Hard NMS** | Binary suppress (remove if IoU > T) | Standard, most common |
| **Soft NMS** | Decay score instead of removing | Overlapping objects (crowds) |
| **DIoU-NMS** | Uses distance-based IoU | Better for overlapping same-class |
| **Weighted NMS** | Merge boxes weighted by score | Higher localization accuracy |
| **Class-Agnostic NMS** | NMS across all classes | When objects can be multi-label |

### Soft NMS:
Instead of removing boxes, reduce their score:
$$s_i = s_i \cdot e^{-\frac{IoU^2}{\sigma}}$$
This is better for crowded scenes where objects genuinely overlap.

## Key Takeaways
- NMS is critical post-processing for most detectors (except NMS-free models like YOLOv10, DETR)
- **IoU threshold** is a key hyperparameter: too high → duplicate boxes, too low → missed objects
- **Soft NMS** is preferred when objects naturally overlap (pedestrian detection, medical imaging)
- Modern trend: NMS-free detection via set-based prediction or dual-head design